# LeetCode 4102 — Count Valid Substrings I (Medium)

Bài toán LeetCode:

```python
class Solution:
    def countValidSubstrings(self, s: str) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4102 (Biweekly Contest 195 — câu 1).

Notebook này dạy kỹ thuật **đếm chuỗi con theo điểm kết thúc**: thay vì thử mọi cặp `(đầu, cuối)`, ta cố định **điểm cuối `r`** và trả lời câu hỏi *"có bao nhiêu điểm đầu hợp lệ?"* chỉ bằng một biến nhớ. Đây là nền tảng của rất nhiều bài "đếm substring / subarray thỏa điều kiện".

---

## Sau notebook này, bạn sẽ nắm được

1. Phân biệt **substring (chuỗi con liên tiếp)** và **subsequence (dãy con)** — điểm mấu chốt để không đọc nhầm đề.
2. Một chuỗi độ dài `n` có bao nhiêu substring và vì sao đếm từng cái là quá chậm.
3. Cách Brute Force `O(n³)`, cải tiến `O(n²)` nhờ **tính chất đơn điệu** ("đã hỏng thì kéo dài vẫn hỏng").
4. Lời giải `O(n)`: cố định điểm cuối, nhớ **vị trí xuất hiện gần nhất của `"abc"`**.
5. Cách **đếm phần bù**: tổng số substring trừ đi số substring "xấu".
6. Các bẫy về chỉ số âm trong Python, lệch một đơn vị, và cách tự kiểm tra bằng test ngẫu nhiên.
7. Mẹo nhận diện, biến thể và bài tập vận dụng.

# 1. Đề bài yêu cầu gì?

Cho chuỗi `s` chỉ gồm các ký tự `'a'`, `'b'`, `'c'`.

Một substring được gọi là **hợp lệ (valid)** nếu nó **không chứa `"abc"` như một substring**, tức là không có ba ký tự **liền nhau** `a`, `b`, `c` theo đúng thứ tự đó.

Yêu cầu: đếm số substring hợp lệ của `s`. Hai substring có vị trí bắt đầu hoặc kết thúc khác nhau thì được tính là khác nhau (kể cả khi nội dung giống nhau).

---

## Ví dụ 1

```python
s = "abac"   ->   10
```

`"abac"` không hề chứa `"abc"` liền nhau, nên **mọi** substring đều hợp lệ:

```text
"a", "ab", "aba", "abac", "b", "ba", "bac", "a", "ac", "c"   -> 10 substring
```

Để ý `"a"` xuất hiện hai lần (vị trí 0 và vị trí 2) và được đếm hai lần.

## Ví dụ 2

```python
s = "abc"   ->   5
```

Có 6 substring: `"a", "ab", "abc", "b", "bc", "c"`. Chỉ `"abc"` là không hợp lệ → còn 5.

## Ví dụ 3

```python
s = "ac"   ->   3
```

Ba substring `"a", "ac", "c"` đều hợp lệ.

---

## Ràng buộc

```text
1 <= s.length <= 10^5
s chỉ gồm 'a', 'b', 'c'
```

# 2. Kiến thức nền tảng

## 2.1. Substring khác subsequence như thế nào?

| Khái niệm | Định nghĩa | Ví dụ với `s = "aXbYc"` |
|---|---|---|
| **substring** (chuỗi con) | các ký tự **liền nhau** | `"aXb"` là substring, `"abc"` **không** phải |
| **subsequence** (dãy con) | giữ thứ tự, **được bỏ bớt** ký tự ở giữa | `"abc"` **là** subsequence (bỏ `X`, `Y`) |

Bài này (phiên bản **I**) nói về **substring**: chỉ cấm `"abc"` **liền nhau**.

> Chú ý: bài 4104 *Count Valid Substrings II* giữ nguyên đề nhưng đổi thành **subsequence**. Đọc nhầm hai chữ này là giải sai cả bài.

Trong Python, `t in s` với hai chuỗi kiểm tra **substring**:

In [1]:
print("abc" in "xxabcxx")    # True: có 'abc' liền nhau
print("abc" in "aXbYc")      # False: a, b, c không liền nhau
print("abc" in "abac")       # False
print("abc" in "cab")        # False

True
False
False
False


## 2.2. Lấy substring bằng slicing

Substring bắt đầu ở chỉ số `i` và kết thúc ở chỉ số `j` (tính cả `j`) là:

```python
s[i:j + 1]
```

Vì slicing `s[a:b]` lấy từ `a` tới **trước** `b`.

Một chuỗi độ dài `n` có bao nhiêu substring? Với mỗi điểm đầu `i` có `n - i` điểm cuối, nên tổng là:

```text
n + (n - 1) + ... + 1 = n * (n + 1) / 2
```

In [2]:
s = "abc"
n = len(s)

for i in range(n):
    for j in range(i, n):
        print(f"s[{i}:{j + 1}] = {s[i:j + 1]!r}")

print("Số substring:", n * (n + 1) // 2)

s[0:1] = 'a'
s[0:2] = 'ab'
s[0:3] = 'abc'
s[1:2] = 'b'
s[1:3] = 'bc'
s[2:3] = 'c'
Số substring: 6


## 2.3. Chỉ số âm trong Python — một cái bẫy

Python cho phép chỉ số âm: `s[-1]` là ký tự **cuối cùng**, `s[-2]` là ký tự kế cuối.

Điều này tiện, nhưng rất nguy hiểm khi ta viết `s[r - 2]` với `r = 0` hoặc `r = 1`: thay vì báo lỗi, Python lặng lẽ **vòng ra cuối chuỗi**.

In [3]:
s = "cab"
r = 0
print("s[r - 2] =", s[r - 2], "| s[r - 1] =", s[r - 1], "| s[r] =", s[r])
# Ở r = 0, ba ký tự "liền trước" lại đọc thành 'a', 'b', 'c' -> nhận nhầm có "abc"!

s[r - 2] = a | s[r - 1] = b | s[r] = c


Ta sẽ quay lại cái bẫy này ở phần lỗi thường gặp. Quy tắc an toàn: **chỉ kiểm tra `s[r-2:r+1] == "abc"` khi `r >= 2`.**

---

# 3. Cách 1 — Brute Force: thử mọi substring

Ý tưởng thẳng nhất: liệt kê mọi cặp `(i, j)`, cắt `s[i:j+1]`, kiểm tra có chứa `"abc"` không.

In [4]:
def count_valid_bruteforce(s: str) -> int:
    n = len(s)
    count = 0
    for i in range(n):
        for j in range(i, n):
            if "abc" not in s[i:j + 1]:
                count += 1
    return count


print(count_valid_bruteforce("abac"))   # 10
print(count_valid_bruteforce("abc"))    # 5
print(count_valid_bruteforce("ac"))     # 3

10
5
3


## Độ phức tạp của Brute Force

- Có khoảng `n²/2` cặp `(i, j)`.
- Mỗi lần cắt `s[i:j+1]` và kiểm tra `in` tốn thêm tới `O(n)`.

```text
Time:  O(n^3)
Space: O(n)   (chuỗi tạm khi cắt)
```

Với `n = 10^5`, riêng số substring đã là khoảng **5 tỷ**:

In [5]:
n = 10**5
print(f"Số substring khi n = 10^5: {n * (n + 1) // 2:,}")

Số substring khi n = 10^5: 5,000,050,000


Không thể kiểm tra từng substring. Ta cần một cách **đếm theo nhóm**.

---

# 4. Cách 2 — Cải tiến O(n²) nhờ tính chất "đã hỏng thì luôn hỏng"

## Quan sát quan trọng (tính đơn điệu)

Cố định điểm đầu `i` và kéo dài điểm cuối `j` sang phải:

```text
s = "aabcab", i = 0
s[0:1] = "a"        hợp lệ
s[0:2] = "aa"       hợp lệ
s[0:3] = "aab"      hợp lệ
s[0:4] = "aabc"     KHÔNG hợp lệ  (chứa "abc")
s[0:5] = "aabca"    KHÔNG hợp lệ
s[0:6] = "aabcab"   KHÔNG hợp lệ
```

> Một khi substring đã chứa `"abc"`, kéo dài thêm vẫn **chứa nguyên** `"abc"` đó → **mãi mãi không hợp lệ**.

Vậy với mỗi `i`, ta kéo `j` sang phải và **dừng ngay** khi lần đầu thấy `"abc"` kết thúc tại `j`. Kiểm tra điều đó chỉ cần nhìn **3 ký tự cuối** `s[j-2:j+1]`, không cần cắt cả substring.

Lưu ý: `"abc"` phải nằm **trọn trong** substring, tức là điểm bắt đầu của nó `j - 2` phải `>= i`.

In [6]:
def count_valid_early_break(s: str) -> int:
    n = len(s)
    count = 0
    for i in range(n):
        for j in range(i, n):
            # "abc" kết thúc tại j và nằm trọn trong s[i..j]?
            if j - 2 >= i and s[j - 2:j + 1] == "abc":
                break              # từ đây trở đi đều không hợp lệ
            count += 1
    return count


print(count_valid_early_break("abac"))   # 10
print(count_valid_early_break("abc"))    # 5
print(count_valid_early_break("ac"))     # 3

10
5
3


## Vẫn chưa đủ nhanh

Nếu chuỗi **không chứa `"abc"`** (ví dụ `"aaaa...a"`), vòng trong không bao giờ `break` sớm → vẫn chạy đủ `n²/2` bước, tức `O(n²)`. Với `n = 10^5` là 5 tỷ bước — vẫn quá chậm.

Cell dưới đếm số bước thực sự của cách 2 trên chuỗi toàn chữ `a` để thấy rõ tốc độ tăng gấp 4 khi `n` tăng gấp đôi.

In [7]:
def steps_early_break(s: str) -> int:
    n = len(s)
    steps = 0
    for i in range(n):
        for j in range(i, n):
            steps += 1
            if j - 2 >= i and s[j - 2:j + 1] == "abc":
                break
    return steps


for n in [500, 1000, 2000]:
    print(f"n = {n:>4}: {steps_early_break('a' * n):>9,} bước")

n =  500:   125,250 bước
n = 1000:   500,500 bước


n = 2000: 2,001,000 bước


# 5. Bước đột phá: cố định ĐIỂM CUỐI thay vì điểm đầu

Đổi góc nhìn: với mỗi **điểm cuối `r`**, hỏi *"có bao nhiêu điểm đầu `i <= r` làm cho `s[i..r]` hợp lệ?"*

Nhờ tính đơn điệu (lần này theo chiều ngược lại: **thu ngắn từ bên trái** thì càng dễ hợp lệ), các điểm đầu hợp lệ luôn là một **dãy liên tiếp sát bên phải**:

```text
i:      0    1    2   ...   p   p+1  ...   r
       xấu  xấu  xấu  ...  xấu  tốt  ...  tốt
```

Vậy điểm ranh giới `p` là gì?

> Gọi `last` = **vị trí bắt đầu của lần xuất hiện `"abc"` gần nhất** mà nằm trọn trong `s[0..r]` (tức là kết thúc tại vị trí `<= r`).
>
> - Nếu `i <= last` thì `s[i..r]` chứa trọn `"abc"` đó → **xấu**.
> - Nếu `i > last` thì `s[i..r]` không chứa trọn bất kỳ `"abc"` nào (vì mọi lần xuất hiện khác còn bắt đầu sớm hơn `last`) → **tốt**.

Số điểm đầu tốt là các `i` từ `last + 1` tới `r`:

```text
số substring hợp lệ kết thúc tại r  =  r - last
```

Nếu chưa gặp `"abc"` nào, đặt `last = -1` → công thức cho `r + 1`, tức là **mọi** điểm đầu đều tốt.

## Ví dụ tính tay với `s = "aabcabca"`

| r | s[r] | `s[r-2..r]` là `"abc"`? | last | r - last |
|---:|:---:|:---:|---:|---:|
| 0 | a | — | -1 | 1 |
| 1 | a | — | -1 | 2 |
| 2 | b | `"aab"` không | -1 | 3 |
| 3 | c | `"abc"` **có** → last = 1 | 1 | 2 |
| 4 | a | `"bca"` không | 1 | 3 |
| 5 | b | `"cab"` không | 1 | 4 |
| 6 | c | `"abc"` **có** → last = 4 | 4 | 2 |
| 7 | a | `"bca"` không | 4 | 3 |

Tổng: `1 + 2 + 3 + 2 + 3 + 4 + 2 + 3 = 20`.

Ví dụ ở `r = 3`: các điểm đầu `i = 0, 1` cho `"aabc"`, `"abc"` (xấu); `i = 2, 3` cho `"bc"`, `"c"` (tốt) → đúng `3 - 1 = 2`.

In [8]:
# Kiểm tra bảng trên bằng brute force: liệt kê điểm đầu tốt cho từng r
s = "aabcabca"
for r in range(len(s)):
    good_starts = [i for i in range(r + 1) if "abc" not in s[i:r + 1]]
    print(f"r = {r}: điểm đầu tốt = {good_starts} -> {len(good_starts)} substring")

r = 0: điểm đầu tốt = [0] -> 1 substring
r = 1: điểm đầu tốt = [0, 1] -> 2 substring
r = 2: điểm đầu tốt = [0, 1, 2] -> 3 substring
r = 3: điểm đầu tốt = [2, 3] -> 2 substring
r = 4: điểm đầu tốt = [2, 3, 4] -> 3 substring
r = 5: điểm đầu tốt = [2, 3, 4, 5] -> 4 substring
r = 6: điểm đầu tốt = [5, 6] -> 2 substring
r = 7: điểm đầu tốt = [5, 6, 7] -> 3 substring


Đúng như dự đoán: điểm đầu tốt luôn là một dãy liên tiếp kết thúc tại `r`.

---

# 6. Cách 3 — Lời giải O(n): một biến `last`

Thuật toán:

1. `last = -1`, `answer = 0`.
2. Duyệt `r` từ `0` tới `n - 1`:
   - Nếu `r >= 2` và `s[r-2:r+1] == "abc"` thì cập nhật `last = r - 2` (vị trí **bắt đầu** của `"abc"`).
   - Cộng `r - last` vào `answer`.
3. Trả về `answer`.

Mỗi vị trí chỉ làm việc hằng số → `O(n)`.

In [9]:
def count_valid_last(s: str) -> int:
    last = -1          # vị trí bắt đầu của "abc" gần nhất; -1 nghĩa là chưa gặp
    answer = 0
    for r in range(len(s)):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r - 2
        answer += r - last
    return answer


print(count_valid_last("abac"))       # 10
print(count_valid_last("abc"))        # 5
print(count_valid_last("ac"))         # 3
print(count_valid_last("aabcabca"))   # 20

10
5
3
20


## Mô phỏng từng bước bằng `print`

Hàm trace in ra mọi thứ cần quan sát: `r`, ký tự, 3 ký tự cuối, `last` trước/sau, số substring hợp lệ kết thúc tại `r`, và tổng tích lũy.

In [10]:
def trace_count_valid(s: str) -> int:
    last = -1
    answer = 0
    print(f"s = {s!r}")
    for r in range(len(s)):
        window = s[r - 2:r + 1] if r >= 2 else "(chưa đủ 3)"
        before = last
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r - 2
        answer += r - last
        note = f"  <- gặp 'abc', last: {before} -> {last}" if last != before else ""
        print(f"r = {r} | s[r] = {s[r]} | 3 ký tự cuối = {window:>11} | "
              f"r - last = {r} - ({last}) = {r - last} | tổng = {answer}{note}")
    return answer


trace_count_valid("aabcabca")

s = 'aabcabca'
r = 0 | s[r] = a | 3 ký tự cuối = (chưa đủ 3) | r - last = 0 - (-1) = 1 | tổng = 1
r = 1 | s[r] = a | 3 ký tự cuối = (chưa đủ 3) | r - last = 1 - (-1) = 2 | tổng = 3
r = 2 | s[r] = b | 3 ký tự cuối =         aab | r - last = 2 - (-1) = 3 | tổng = 6
r = 3 | s[r] = c | 3 ký tự cuối =         abc | r - last = 3 - (1) = 2 | tổng = 8  <- gặp 'abc', last: -1 -> 1
r = 4 | s[r] = a | 3 ký tự cuối =         bca | r - last = 4 - (1) = 3 | tổng = 11
r = 5 | s[r] = b | 3 ký tự cuối =         cab | r - last = 5 - (1) = 4 | tổng = 15
r = 6 | s[r] = c | 3 ký tự cuối =         abc | r - last = 6 - (4) = 2 | tổng = 17  <- gặp 'abc', last: 1 -> 4
r = 7 | s[r] = a | 3 ký tự cuối =         bca | r - last = 7 - (4) = 3 | tổng = 20


20

# 7. Cách 4 — Đếm phần bù: tổng trừ "xấu"

Nhiều bài đếm dễ hơn nếu ta đếm **cái ngược lại**.

- Tổng số substring: `n * (n + 1) / 2`.
- Với mỗi điểm cuối `r`, số điểm đầu **xấu** là các `i` từ `0` tới `last`, tức là `last + 1` điểm (bằng `0` nếu `last = -1`).

```text
answer = n(n + 1)/2  -  Σ (last_r + 1)
```

Đây chính là cách 3 nhìn từ phía ngược lại, nhưng việc tập thói quen **"đếm phần bù"** rất có ích: nhiều bài "đếm số substring chứa ít nhất một ..." được giải nhanh theo cách này.

In [11]:
def count_valid_complement(s: str) -> int:
    n = len(s)
    total = n * (n + 1) // 2
    bad = 0
    last = -1
    for r in range(n):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r - 2
        bad += last + 1            # các điểm đầu 0..last đều xấu
    return total - bad


for s in ["abac", "abc", "ac", "aabcabca"]:
    n = len(s)
    print(f"{s!r:11}: tổng = {n * (n + 1) // 2:>2}, hợp lệ = {count_valid_complement(s)}")

'abac'     : tổng = 10, hợp lệ = 10
'abc'      : tổng =  6, hợp lệ = 5
'ac'       : tổng =  3, hợp lệ = 3
'aabcabca' : tổng = 36, hợp lệ = 20


# 8. Phiên bản đúng format LeetCode

Ta nộp cách 3. Có thể viết gọn điều kiện `s[r-2:r+1] == "abc"` thành so sánh từng ký tự để tránh tạo chuỗi tạm (nhanh hơn một chút).

In [12]:
class Solution:
    def countValidSubstrings(self, s: str) -> int:
        last = -1
        answer = 0
        for r in range(len(s)):
            if r >= 2 and s[r] == 'c' and s[r - 1] == 'b' and s[r - 2] == 'a':
                last = r - 2
            answer += r - last
        return answer

In [13]:
sol = Solution()

tests = [
    ("abac", 10, "ví dụ 1"),
    ("abc", 5, "ví dụ 2"),
    ("ac", 3, "ví dụ 3"),
    ("a", 1, "một ký tự"),
    ("cab", 6, "bẫy chỉ số âm"),
    ("abcabc", 14, "hai lần 'abc' sát nhau"),
    ("aaaa", 10, "không có 'abc'"),
    ("aabcabca", 20, "ví dụ tự tính"),
]

for s, expected, name in tests:
    result = sol.countValidSubstrings(s)
    assert result == expected, (name, s, result, expected)
    print(f"PASS | {name:24} | s = {s!r:11} -> {result}")

PASS | ví dụ 1                  | s = 'abac'      -> 10
PASS | ví dụ 2                  | s = 'abc'       -> 5
PASS | ví dụ 3                  | s = 'ac'        -> 3
PASS | một ký tự                | s = 'a'         -> 1
PASS | bẫy chỉ số âm            | s = 'cab'       -> 6
PASS | hai lần 'abc' sát nhau   | s = 'abcabc'    -> 14
PASS | không có 'abc'           | s = 'aaaa'      -> 10
PASS | ví dụ tự tính            | s = 'aabcabca'  -> 20


Kiểm tra với chuỗi dài `10^5` ký tự để chắc chắn đủ nhanh:

In [14]:
import random
import time

random.seed(4102)
big = "".join(random.choice("abc") for _ in range(10**5))

start = time.perf_counter()
result = sol.countValidSubstrings(big)
elapsed = time.perf_counter() - start
print(f"n = {len(big):,} -> kết quả {result:,}, chạy trong {elapsed:.3f} giây")

n = 100,000 -> kết quả 2,543,989, chạy trong 0.008 giây


# 9. Vì sao thuật toán đúng?

Ta cần chứng minh: với mỗi `r`, số substring hợp lệ kết thúc tại `r` đúng bằng `r - last`.

**(1) Mọi `i <= last` đều xấu.** Lần xuất hiện `"abc"` tại `last..last+2` có `last + 2 <= r`, và `i <= last`, nên nó nằm trọn trong `s[i..r]`.

**(2) Mọi `i > last` đều tốt.** Giả sử `s[i..r]` chứa `"abc"` tại vị trí `p..p+2` với `i <= p` và `p + 2 <= r`. Vì `last` là vị trí bắt đầu **lớn nhất** trong các lần xuất hiện kết thúc tại `<= r`, ta có `p <= last < i` — mâu thuẫn với `i <= p`.

**(3) `last` luôn được cập nhật đúng.** Một lần xuất hiện mới kết thúc tại `r` thì bắt đầu tại `r - 2`, lớn hơn mọi vị trí bắt đầu trước đó → nó trở thành `last` mới. Nếu không có lần mới, `last` giữ nguyên.

Cộng tất cả `r` lại, mỗi substring `s[i..r]` được đếm đúng một lần (theo điểm cuối của nó) → tổng đúng.

---

## Trực giác

> Cái "chướng ngại" gần nhất bên trái quyết định tất cả. Điểm đầu nào nằm **sau** chướng ngại gần nhất thì an toàn; nằm **trước hoặc tại** nó thì bị dính.

# 10. Độ phức tạp

Gọi `n = len(s)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | mọi `(i, j)`, cắt rồi kiểm tra | `O(n³)` | `O(n)` |
| 2. Dừng sớm | cố định `i`, kéo `j`, gặp `"abc"` thì dừng | `O(n²)` | `O(1)` |
| 3. Cố định điểm cuối | một biến `last` | `O(n)` | `O(1)` |
| 4. Đếm phần bù | tổng trừ số xấu | `O(n)` | `O(1)` |

Ghi chú: kết quả có thể tới `n(n+1)/2 ≈ 5·10^9` khi `n = 10^5`. Python tự xử lý số lớn; trong C++/Java phải dùng kiểu `long long` / `long`.

# 11. Đối chiếu các cách trên dữ liệu ngẫu nhiên

In [15]:
random.seed(2026)

for _ in range(2000):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 14)))
    expected = count_valid_bruteforce(s)
    assert count_valid_early_break(s) == expected, s
    assert count_valid_last(s) == expected, s
    assert count_valid_complement(s) == expected, s
    assert sol.countValidSubstrings(s) == expected, s

print("2000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

2000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


# 12. Những lỗi dễ mắc

## Lỗi 1 — Kiểm tra `s[r-2], s[r-1], s[r]` mà quên điều kiện `r >= 2`

Vì chỉ số âm vòng ra cuối chuỗi (mục 2.3), với `s = "cab"` code sai sẽ "nhìn thấy" `"abc"` ở `r = 0`.

In [16]:
def wrong_negative_index(s: str) -> int:
    last = -1
    answer = 0
    for r in range(len(s)):
        if s[r - 2] == 'a' and s[r - 1] == 'b' and s[r] == 'c':   # thiếu r >= 2
            last = r - 2
        answer += r - last
    return answer


print("Sai :", wrong_negative_index("cab"))
print("Đúng:", sol.countValidSubstrings("cab"))

Sai : 9
Đúng: 6


Ở `r = 0` code sai đặt `last = -2`, khiến `r - last = 2` dù chỉ có **một** substring kết thúc tại `0`. Kết quả bị đếm thừa.

(Cách viết slicing `s[r-2:r+1] == "abc"` ít bị bẫy hơn vì `s[-2:1]` thường cho chuỗi rỗng, nhưng hãy tập thói quen ghi rõ `r >= 2` để code tự giải thích.)

---

## Lỗi 2 — Lưu vị trí **kết thúc** thay vì vị trí **bắt đầu**

Code sai:

```python
if r >= 2 and s[r-2:r+1] == "abc":
    last = r          # SAI: phải là r - 2
```

Với `s = "abc"`, ở `r = 2` code sai cho `r - last = 0`, nhưng thực tế còn `"bc"` và `"c"` hợp lệ (2 substring).

In [17]:
def wrong_store_end(s: str) -> int:
    last = -1
    answer = 0
    for r in range(len(s)):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r                     # SAI
        answer += r - last
    return answer


print("Sai :", wrong_store_end("abc"))
print("Đúng:", sol.countValidSubstrings("abc"))

Sai : 3
Đúng: 5


## Lỗi 3 — Cộng `r - last + 1` (lệch một đơn vị)

Các điểm đầu tốt là `last + 1, last + 2, ..., r`. Số phần tử của dãy số nguyên từ `a` tới `b` là `b - a + 1`, với `a = last + 1`, `b = r` cho ra `r - last`. Mẹo: luôn kiểm tra công thức với trường hợp nhỏ nhất, như `s = "a"` phải ra `1`.

---

## Lỗi 4 — Đọc nhầm "substring" thành "subsequence"

Nếu hiểu là subsequence, `"abac"` cũng chứa `a, b, c` theo thứ tự (`a` ở 0, `b` ở 1, `c` ở 3) → đáp án sẽ là `9` chứ không phải `10`. Đó là đề của bài **4104**. Luôn kiểm tra lại đề bằng các ví dụ cho sẵn.

---

## Lỗi 5 — Brute Force cắt chuỗi trong hai vòng lặp

`O(n³)` chạy đúng với ví dụ nhỏ nhưng TLE ngay khi `n` vài nghìn. Hãy ước lượng số phép toán từ constraints **trước** khi code.

# 13. Mẹo nhận diện

Nghĩ tới **"cố định điểm cuối, nhớ ranh giới trái"** khi:

1. Đề yêu cầu **đếm số substring / subarray** thỏa một điều kiện.
2. Điều kiện có **tính đơn điệu**: nếu `s[i..r]` xấu thì mọi đoạn **chứa nó** cũng xấu (tương đương: nếu tốt thì mọi đoạn con của nó cũng tốt).
3. Khi đó, với mỗi `r`, các điểm đầu tốt là một dãy liên tiếp `[left, r]` → cộng `r - left + 1`.

Công thức vàng của họ bài này:

```python
for r in range(n):
    # cập nhật ranh giới trái nhỏ nhất còn tốt: left
    answer += r - left + 1
```

Ở bài này `left = last + 1`, nên `r - left + 1 = r - last`.

Các bài cùng họ: *Count Substrings That Satisfy K-Constraint*, *Number of Substrings Containing All Three Characters* (1358), *Subarray Product Less Than K* (713), *Count Subarrays With Fixed Bounds* (2444).

# 14. Biến thể

## Biến thể 1 — Mẫu cấm là một chuỗi `t` bất kỳ

Thay `"abc"` bằng chuỗi `t` độ dài `m`. Ý tưởng y hệt: khi `t` kết thúc tại `r` thì `last = r - m + 1`.

In [18]:
def count_without_pattern(s: str, t: str) -> int:
    m = len(t)
    last = -1
    answer = 0
    for r in range(len(s)):
        if r >= m - 1 and s[r - m + 1:r + 1] == t:
            last = r - m + 1
        answer += r - last
    return answer


print(count_without_pattern("abac", "abc"))     # 10, giống bài gốc
print(count_without_pattern("aaaa", "aa"))      # chỉ còn 4 substring "a"
print(count_without_pattern("abab", "ba"))      # "a","ab","b","a","ab","b"... tự kiểm tra

10
4
6


Với `t` dài, so sánh `s[r-m+1:r+1] == t` tốn `O(m)` mỗi bước → `O(n·m)`. Có thể tăng tốc bằng thuật toán KMP hoặc Rolling Hash, nhưng ý tưởng đếm theo điểm cuối không đổi.

## Biến thể 2 — Độ dài substring hợp lệ **dài nhất**

Với mỗi `r`, substring hợp lệ dài nhất kết thúc tại `r` là `s[last+1..r]`, độ dài `r - last`. Lấy max.

In [19]:
def longest_valid(s: str) -> int:
    last = -1
    best = 0
    for r in range(len(s)):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r - 2
        best = max(best, r - last)
    return best


print(longest_valid("abac"))       # 4: cả chuỗi
print(longest_valid("aabcabca"))   # 4: ví dụ "bcab"
print(longest_valid("abc"))        # 2

4
4
2


# 15. Template tư duy

```python
last = -1                      # vị trí bắt đầu của "chướng ngại" gần nhất
answer = 0
for r in range(n):
    if <một chướng ngại mới kết thúc tại r>:
        last = <vị trí BẮT ĐẦU của chướng ngại đó>
    answer += r - last         # số điểm đầu i trong [last + 1, r]
```

Phiên bản đếm phần bù:

```python
answer = n * (n + 1) // 2 - sum(last_r + 1 for r in range(n))
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Lập bảng `r | s[r] | last | r - last` cho:

```python
s = "babcbc"
```

Kết quả là bao nhiêu? Kiểm tra bằng cell dưới.

In [20]:
# Kiểm tra Bài 1 sau khi đã tính tay
trace_count_valid("babcbc");

s = 'babcbc'
r = 0 | s[r] = b | 3 ký tự cuối = (chưa đủ 3) | r - last = 0 - (-1) = 1 | tổng = 1
r = 1 | s[r] = a | 3 ký tự cuối = (chưa đủ 3) | r - last = 1 - (-1) = 2 | tổng = 3
r = 2 | s[r] = b | 3 ký tự cuối =         bab | r - last = 2 - (-1) = 3 | tổng = 6
r = 3 | s[r] = c | 3 ký tự cuối =         abc | r - last = 3 - (1) = 2 | tổng = 8  <- gặp 'abc', last: -1 -> 1
r = 4 | s[r] = b | 3 ký tự cuối =         bcb | r - last = 4 - (1) = 3 | tổng = 11
r = 5 | s[r] = c | 3 ký tự cuối =         cbc | r - last = 5 - (1) = 4 | tổng = 15


## Bài 2 — Dự đoán output

Đoán kết quả từng dòng trước khi chạy:

```python
s = "abcab"
print(s[-1], s[-2])
print(s[1:4])
print(s[-2:1])
print(len(s) * (len(s) + 1) // 2)
```

In [21]:
s = "abcab"
print(s[-1], s[-2])
print(s[1:4])
print(repr(s[-2:1]))
print(len(s) * (len(s) + 1) // 2)

b a
bca
''
15


## Bài 3 — Hoàn thiện lời giải chính

Điền các chỗ `TODO`.

In [22]:
def exercise_3_count_valid(s: str) -> int:
    last = -1
    answer = 0
    for r in range(len(s)):
        # TODO 1: nếu "abc" kết thúc tại r (nhớ điều kiện r >= 2) thì cập nhật last
        # TODO 2: cộng số substring hợp lệ kết thúc tại r vào answer
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_count_valid("abac") == 10
# assert exercise_3_count_valid("abc") == 5
# assert exercise_3_count_valid("cab") == 6

## Bài 4 — Đếm substring **chứa** `"abc"` ít nhất một lần

Viết hàm trả về số substring có chứa `"abc"` (liền nhau). Thử viết **không** dùng hàm của bài gốc.

```python
count_containing("abc")       -> 1
count_containing("aabcabca")  -> 16
```

In [23]:
def exercise_4_count_containing(s: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_count_containing("abc") == 1
# assert exercise_4_count_containing("aabcabca") == 16
# assert exercise_4_count_containing("abac") == 0

## Bài 5 — Không được có hai ký tự giống nhau liền kề

Đếm số substring **không** có hai ký tự giống nhau đứng cạnh nhau (ví dụ `"abca"` tốt, `"abba"` xấu).

```python
count_no_equal_adjacent("aab")   -> 4    # "a", "a", "ab", "b"
count_no_equal_adjacent("abc")   -> 6
```

Gợi ý: "chướng ngại" ở đây là cặp `s[r-1] == s[r]`.

In [24]:
def exercise_5_count_no_equal_adjacent(s: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_no_equal_adjacent("aab") == 4
# assert exercise_5_count_no_equal_adjacent("abc") == 6
# assert exercise_5_count_no_equal_adjacent("aaaa") == 4

## Bài 6 — Mini challenge: chứa `"abc"` **đúng một lần**

Đếm số substring chứa `"abc"` (liền nhau) **đúng một lần**.

```python
count_exactly_one("abc")       -> 1
count_exactly_one("abcabc")    -> 6
count_exactly_one("aabcabca")  -> 12
```

Gợi ý lớn: ngoài `last` (lần xuất hiện gần nhất), hãy nhớ thêm `prev` (lần xuất hiện **trước đó nữa**).

In [25]:
def exercise_6_count_exactly_one(s: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_count_exactly_one("abc") == 1
# assert exercise_6_count_exactly_one("abcabc") == 6
# assert exercise_6_count_exactly_one("aabcabca") == 12

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: nhìn 3 ký tự cuối khi `r >= 2`.
- Mức 2: `if r >= 2 and s[r-2:r+1] == "abc": last = r - 2`.
- Mức 3: `answer += r - last`, cuối hàm `return answer`.

**Bài 4**

- Mức 1: số substring **xấu** kết thúc tại `r` là bao nhiêu?
- Mức 2: là `last + 1` (các điểm đầu `0..last`).
- Mức 3: hoặc lấy tổng `n(n+1)/2` trừ kết quả bài gốc.

**Bài 5**

- Mức 1: khi `s[r-1] == s[r]`, mọi substring kết thúc tại `r` mà bắt đầu `<= r - 1` đều xấu.
- Mức 2: vậy "chướng ngại" bắt đầu ở `r - 1` → `last = r - 1`.
- Mức 3: `answer += r - last`.

**Bài 6**

- Mức 1: substring `s[i..r]` chứa đúng một lần khi nó chứa lần `last` nhưng **không** chứa lần `prev`.
- Mức 2: điều đó tương đương `prev < i <= last`.
- Mức 3: cộng `last - prev` với mỗi `r` (khi `last = -1` thì cộng `0`). Khi gặp lần mới: `prev, last = last, r - 2`.

# 17. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

`s = "babcbc"`: chỉ có một lần `"abc"` tại vị trí `1..3`.

| r | s[r] | last | r - last |
|---:|:---:|---:|---:|
| 0 | b | -1 | 1 |
| 1 | a | -1 | 2 |
| 2 | b | -1 | 3 |
| 3 | c | 1 | 2 |
| 4 | b | 1 | 3 |
| 5 | c | 1 | 4 |

Tổng: `1 + 2 + 3 + 2 + 3 + 4 = 15`.

## Đáp án Bài 2

```text
b a
bca
''        (s[-2:1] tức s[3:1] -> rỗng)
15
```

In [26]:
# Đáp án Bài 3
def answer_3_count_valid(s: str) -> int:
    last = -1
    answer = 0
    for r in range(len(s)):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r - 2
        answer += r - last
    return answer


assert answer_3_count_valid("abac") == 10
assert answer_3_count_valid("abc") == 5
assert answer_3_count_valid("cab") == 6
assert answer_3_count_valid("babcbc") == 15
print("Bài 3: PASS")

Bài 3: PASS


In [27]:
# Đáp án Bài 4
def answer_4_count_containing(s: str) -> int:
    last = -1
    bad = 0
    for r in range(len(s)):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r - 2
        bad += last + 1
    return bad


assert answer_4_count_containing("abc") == 1
assert answer_4_count_containing("aabcabca") == 16
assert answer_4_count_containing("abac") == 0
print("Bài 4: PASS")

Bài 4: PASS


In [28]:
# Đáp án Bài 5
def answer_5_count_no_equal_adjacent(s: str) -> int:
    last = -1
    answer = 0
    for r in range(len(s)):
        if r >= 1 and s[r - 1] == s[r]:
            last = r - 1
        answer += r - last
    return answer


assert answer_5_count_no_equal_adjacent("aab") == 4
assert answer_5_count_no_equal_adjacent("abc") == 6
assert answer_5_count_no_equal_adjacent("aaaa") == 4
print("Bài 5: PASS")

Bài 5: PASS


In [29]:
# Đáp án Bài 6
def answer_6_count_exactly_one(s: str) -> int:
    prev = -1      # lần xuất hiện trước lần gần nhất
    last = -1      # lần xuất hiện gần nhất
    answer = 0
    for r in range(len(s)):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            prev, last = last, r - 2
        answer += last - prev          # các điểm đầu i với prev < i <= last
    return answer


def brute_exactly_one(s: str) -> int:
    n = len(s)
    total = 0
    for i in range(n):
        for j in range(i, n):
            sub = s[i:j + 1]
            occurrences = sum(1 for p in range(len(sub) - 2) if sub[p:p + 3] == "abc")
            total += occurrences == 1
    return total


assert answer_6_count_exactly_one("abc") == 1
assert answer_6_count_exactly_one("abcabc") == 6
assert answer_6_count_exactly_one("aabcabca") == 12
for _ in range(500):
    t = "".join(random.choice("abc") for _ in range(random.randint(1, 12)))
    assert answer_6_count_exactly_one(t) == brute_exactly_one(t), t
print("Bài 6: PASS (kể cả 500 test ngẫu nhiên)")

Bài 6: PASS (kể cả 500 test ngẫu nhiên)


# 18. Tổng kết

Bài **Count Valid Substrings I** dạy một kỹ thuật đếm cực kỳ phổ biến:

> **Cố định điểm cuối `r`.** Nếu điều kiện có tính đơn điệu, các điểm đầu hợp lệ là một dãy liên tiếp `[left, r]`. Chỉ cần duy trì `left` và cộng `r - left + 1`.

Ở bài này, `left = last + 1` với `last` là vị trí bắt đầu của `"abc"` gần nhất.

## Công thức nhớ nhanh

```text
gặp "abc" kết thúc tại r  ->  last = r - 2
mỗi r                     ->  answer += r - last
```

## Phiên bản nên nhớ

```python
class Solution:
    def countValidSubstrings(self, s: str) -> int:
        last = -1
        answer = 0
        for r in range(len(s)):
            if r >= 2 and s[r - 2:r + 1] == "abc":
                last = r - 2
            answer += r - last
        return answer
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(1)
```

# 19. Checklist trước khi submit

- [ ] Đề nói **substring** (liền nhau) hay **subsequence** (được bỏ ký tự)? Bài này là substring.
- [ ] Có điều kiện `r >= 2` trước khi nhìn `s[r-2]` để tránh chỉ số âm vòng ra cuối chuỗi?
- [ ] `last` lưu vị trí **bắt đầu** `r - 2`, không phải vị trí kết thúc?
- [ ] Công thức cộng là `r - last` (không phải `r - last + 1`)? Thử lại với `s = "a"` → `1`.
- [ ] Khởi tạo `last = -1` để khi chưa gặp `"abc"` thì mọi điểm đầu đều hợp lệ?
- [ ] Đã thử các test biên: một ký tự, `"abc"`, `"abcabc"`, chuỗi không có `"abc"`?
- [ ] (Nếu dùng C++/Java) biến kết quả là `long long` / `long`?